In [1]:
# 1. Systeem dependencies
!apt-get update && apt-get install -y zstd

# 2. Ollama installeren
!curl -fsSL https://ollama.com/install.sh | sh

# 3. CrewAI installeren (we negeren de errors van de Google-pakketten)
!pip install -q --no-warn-conflicts crewai langchain_community

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.9 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,615 k

In [2]:
import crewai
import langchain_community
print("CrewAI is succesvol geladen!")

CrewAI is succesvol geladen!


In [3]:
import os
import subprocess
import time

# 1. Installeer zstd en Ollama (met forcering van het pad)
print("Bezig met installeren van dependencies...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Definieer het volledige pad naar ollama
# De installer zet hem meestal in /usr/local/bin/ollama
OLLAMA_PATH = "/usr/local/bin/ollama"

# 3. Start de server op de achtergrond
print("Ollama server opstarten...")
subprocess.Popen([OLLAMA_PATH, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15) # Geef de server tijd

# 4. Pull het model met het volledige pad
print("Model downloaden (kan even duren)...")
subprocess.run([OLLAMA_PATH, "pull", "llama3"])

# 5. Controleer of het model er staat
print("\nGeïnstalleerde modellen:")
subprocess.run([OLLAMA_PATH, "list"])

Bezig met installeren van dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 131 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏ 120 KB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  42 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  63 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   3% ▕                  ▏ 117 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   4% ▕                  ▏ 179 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   4% ▕                  ▏ 206 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   6% ▕█                 ▏ 269 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   7% ▕█                 ▏ 318 MB/4


Geïnstalleerde modellen:
NAME             ID              SIZE      MODIFIED               
llama3:latest    365c0bd3c000    4.7 GB    Less than a second ago    


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

In [4]:
import os
from crewai import Agent, Task, Crew

# 1. Fake OpenAI key (verplicht voor CrewAI)
os.environ["OPENAI_API_KEY"] = "sk-ollama"

# 2. Configureer de Agent met de 'openai' provider stijl, maar wijs naar Ollama
# We gebruiken 'ollama/' als prefix voor de modelnaam
test_agent = Agent(
    role='Kaggle Expert',
    goal='Laat zien dat de 404 error weg is.',
    backstory='Ik ben een AI die lokaal draait.',
    # De magie zit hier:
    llm="ollama/llama3", 
    # We vertellen CrewAI expliciet waar de lokale server staat
    base_url="http://localhost:11434/v1", 
    verbose=True,
    allow_delegation=False
)

# 3. Simpele taak
test_task = Task(
    description='Zeg alleen: "De 404 is opgelost!"',
    expected_output='Een korte bevestiging.',
    agent=test_agent
)

crew = Crew(agents=[test_agent], tasks=[test_task])

print("\n--- START TEST ---")
result = crew.kickoff()
print(result)


--- START TEST ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kaggle Expert                                                                                           │
│                                                                                                                 │
│  Task: Zeg alleen: "De 404 is opgelost!"                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kaggle Expert                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  De 404 is opgelost!                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

De 404 is opgelost!


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

In [5]:
import os
from crewai import Agent, Task, Crew

# 1. Omzeil de OpenAI check
os.environ["OPENAI_API_KEY"] = "sk-ollama"

# Agent 1: De Analist
analist = Agent(
    role='Data Analist',
    goal='Bedenk 3 creatieve namen voor een nieuwe AI startup.',
    backstory='Je bent een creatief brein gespecialiseerd in branding.',
    llm="ollama/llama3",
    base_url="http://localhost:11434/v1",
    allow_delegation=False,
    verbose=True
)

# Agent 2: De Criticus
criticus = Agent(
    role='Marketing Expert',
    goal='Kies de beste naam uit de lijst en leg uit waarom.',
    backstory='Je hebt een scherp oog voor marketingpotentieel.',
    llm="ollama/llama3",
    base_url="http://localhost:11434/v1",
    allow_delegation=False,
    verbose=True
)

# Taken
taak1 = Task(description="Genereer 3 namen voor een AI bedrijf.", agent=analist, expected_output="Een lijst van 3 namen.")
taak2 = Task(description="Kies de beste naam.", agent=criticus, expected_output="De winnende naam met motivatie.")

# De Crew
marketing_crew = Crew(agents=[analist, criticus], tasks=[taak1, taak2])
marketing_crew.kickoff()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analist                                                                                            │
│                                                                                                                 │
│  Task: Genereer 3 namen voor een AI bedrijf.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analist                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  What an exciting task! As a Data Analist, I'm thrilled to generate three creative names for a new AI startup.  │
│  Here are my suggestions:                                                                                       │
│                                                                                                                 │
│  **1.** **CerebroX**                                                                                            │
│  CerebroX combines the Spanish word for "brain" (cerebro) with the suffix "X" to convey innovation and          │
│  experimentation. This name highlights the startup's focus on artificial intelligence and its potential to      │
│  revolutionize industries.                                                                                      │
│                                                                                                                 │
│  **2.** **MindSpark AI**                                                                                        │
│  MindSpark is a playful name that evokes the idea of sparking creativity and innovation. The "AI" suffix        │
│  explicitly communicates the company's area of expertise. This name is catchy and memorable, making it          │
│  suitable for a brand that aims to energize and inspire its customers.                                          │
│                                                                                                                 │
│  **3.** **Nexa Intellect**                                                                                      │
│  Nexa Intellect combines "nexus" (connection) and "intellect" (intelligence) to create a name that emphasizes   │
│  the AI startup's ability to bridge the gap between human understanding and technological innovation. This      │
│  name conveys a sense of sophistication and expertise, making it suitable for a brand that aims to provide      │
│  cutting-edge AI solutions.                                                                                     │
│                                                                                                                 │
│  There you have it! Three creative names that I hope will inspire your new AI startup.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Expert                                                                                        │
│                                                                                                                 │
│  Task: Kies de beste naam.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Expert                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  What an exciting task indeed!                                                                                  │
│                                                                                                                 │
│  After carefully reviewing the three creative names, I am thrilled to announce that I have chosen **CerebroX**  │
│  as the winning name.                                                                                           │
│                                                                                                                 │
│  Here's why:                                                                                                    │
│                                                                                                                 │
│  CerebroX effectively communicates the AI startup's focus on artificial intelligence and its potential to       │
│  revolutionize industries. The combination of the Spanish word for "brain" (cerebro) with the suffix "X"        │
│  conveys innovation and experimentation, which is perfectly in line with the startup's mission. This name is    │
│  not only catchy and memorable but also has a certain edge to it, which will likely resonate with the target    │
│  audience.                                                                                                      │
│                                                                                                                 │
│  While MindSpark AI is a great name that evokes creativity and innovation, it may not explicitly convey the AI  │
│  aspect, which could lead to confusion. Nexa Intellect is a sophisticated name that conveys expertise, but it   │
│  may not be as memorable or attention-grabbing as CerebroX.                                                     │
│                                                                                                                 │
│  In conclusion, CerebroX stands out as the best name for this AI startup due to its unique combination of a     │
│  familiar word with a cutting-edge suffix, which effectively communicates the company's mission and values.     │
│  This name has the potential to spark interest and generate excitement among potential customers, partners,     │
│  and investors.                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

CrewOutput(raw='What an exciting task indeed!\n\nAfter carefully reviewing the three creative names, I am thrilled to announce that I have chosen **CerebroX** as the winning name.\n\nHere\'s why:\n\nCerebroX effectively communicates the AI startup\'s focus on artificial intelligence and its potential to revolutionize industries. The combination of the Spanish word for "brain" (cerebro) with the suffix "X" conveys innovation and experimentation, which is perfectly in line with the startup\'s mission. This name is not only catchy and memorable but also has a certain edge to it, which will likely resonate with the target audience.\n\nWhile MindSpark AI is a great name that evokes creativity and innovation, it may not explicitly convey the AI aspect, which could lead to confusion. Nexa Intellect is a sophisticated name that conveys expertise, but it may not be as memorable or attention-grabbing as CerebroX.\n\nIn conclusion, CerebroX stands out as the best name for this AI startup due to i